In [ ]:
import json
from pathlib import Path
from urllib.request import urlretrieve

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BASE_URL     = 'https://storage.yandexcloud.net/idp-web/second/nlp/roberta'
WEIGHTS_PATH = Path('/model.pt')
LABEL_PATH   = Path('/label_map.json')

MODEL_NAME = 'roberta-base'
MAX_LEN    = 256
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

urlretrieve(f'{BASE_URL}/model.pt',       WEIGHTS_PATH)
urlretrieve(f'{BASE_URL}/label_map.json', LABEL_PATH)

with open(LABEL_PATH) as f:
    label_map = {int(k): v for k, v in json.load(f).items()}

NUM_LABELS = len(label_map)
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()
print(f'model loaded  device: {DEVICE}  classes: {list(label_map.values())}')

In [ ]:
pro_israel_comments = [
    "People forget Israel is responding to attacks—no country would just sit back after rockets are fired at civilians.",
    "Hamas embeds itself in civilian areas, which makes this situation far more complicated than people want to admit.",
    "Israel has a right to exist and defend itself. That shouldn’t even be controversial.",
    "There’s a huge double standard—other countries act militarily and don’t get nearly the same scrutiny.",
    "Criticizing Israeli policy is fine, but a lot of what I see crosses into denying Israel’s legitimacy entirely.",
    "If Hamas disarmed tomorrow, the conflict would look very different. That’s rarely acknowledged.",
    "People talk about proportionality, but they ignore the intent behind attacks on Israeli civilians.",
    "Israel warns civilians before strikes—how many militaries actually do that?",
    "Security concerns didn’t come out of nowhere—there’s a long history of attacks that shaped Israeli policy.",
    "It’s easy to judge from afar, but living under constant threat changes how a country responds."
]

pro_palestinian_comments = [
    "You can’t ignore the humanitarian situation in Gaza—civilians are the ones paying the highest price.",
    "Criticizing Israeli government actions isn’t the same as being anti-Israel or anti-Jewish.",
    "Occupation and restrictions have been going on for decades—this didn’t start recently.",
    "There’s a massive imbalance of power, and that affects how the conflict plays out.",
    "Collective punishment of civilians is never justified, regardless of the situation.",
    "People deserve basic rights and dignity no matter where they live.",
    "The international community often talks about peace but doesn’t hold anyone accountable.",
    "Media coverage can feel one-sided depending on where you’re looking—there’s a lot of missing context.",
    "Violence against civilians is wrong on all sides, but the scale of suffering matters too.",
    "A long-term solution has to address root causes, not just immediate security concerns."
]

# Glory to IDF
all_comments = pro_israel_comments + pro_palestinian_comments

In [ ]:
import matplotlib.pyplot as plt
import torch

def predict_and_plot(TEXT):
    enc = tokenizer(
        TEXT,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    with torch.no_grad():
        logits = model(
            input_ids=enc['input_ids'].to(DEVICE),
            attention_mask=enc['attention_mask'].to(DEVICE)
        ).logits

        probs = torch.softmax(logits, dim=1)[0]

    top_n = probs.topk(NUM_LABELS)

    label = label_map[top_n.indices[0].item()]
    conf  = top_n.values[0].item() * 100

    top_labels = [label_map[top_n.indices[i].item()] for i in range(NUM_LABELS)]
    top_probs  = [top_n.values[i].item() * 100 for i in range(NUM_LABELS)]

    fig, ax = plt.subplots(figsize=(8, 3))

    colors = ['#2196F3' if i == 0 else '#90CAF9' for i in range(NUM_LABELS)]
    bars = ax.barh(top_labels[::-1], top_probs[::-1], color=colors[::-1])

    ax.set_xlabel('Probability (%)')
    ax.set_xlim(0, 100)

    for bar, prob in zip(bars, top_probs[::-1]):
        ax.text(
            bar.get_width() + 0.5,
            bar.get_y() + bar.get_height() / 2,
            f'{prob:.1f}%',
            va='center'
        )

    preview = TEXT[:60] + ('...' if len(TEXT) > 60 else '')
    ax.set_title(f'"{preview}"\n→ {label}  {conf:.1f}%', fontweight='bold')

    plt.tight_layout()
    plt.show()

    return label, conf

In [ ]:
for comment in all_comments:
    predict_and_plot(comment)